# Linear and logistic regression

**CSCI 5612 - Module 4, Project Part 4**

Linear regression predicts a vehicle's crash rate (a number). Logistic regression classifies a single complaint as Crash or No crash from its narrative, on exactly the same data and split as the Naive Bayes notebook. Settings are chosen by cross-validation inside the training set; the test set is used once.

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, mean_absolute_error,
                             mean_squared_error, precision_score, r2_score, recall_score)
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_predict, train_test_split
from sklearn.naive_bayes import BernoulliNB, MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option('display.width', 200)
RANDOM_STATE, TEST_SIZE = 5612, 0.25

## 1. Linear regression data

One row per model-year with at least 20 complaints. The target is the percentage of complaints involving a crash. Body segment is one-hot encoded with SUV as the reference level.

In [2]:
d = pd.read_csv('../data/clean/reg_linear_data.csv')
LAB = ['OverallRating','OverallFrontCrashRating','OverallSideCrashRating','RolloverRating','rollover_prob_pct']
EQUIP = ['has_fcw','has_ldw']
SEG = [c for c in d.columns if c.startswith('seg_')]
RECORD = ['mean_vehicle_age','median_days_to_report','log_complaints','log_recalls']
print(d.shape, '| target mean %.2f sd %.2f' % (d.crash_rate_pct.mean(), d.crash_rate_pct.std()))
d[['make','model','model_year','segment'] + LAB[:2] + RECORD[:2] + ['crash_rate_pct']].head()

(218, 22) | target mean 6.47 sd 5.09


,make,model,model_year,segment,OverallRating,OverallFrontCrashRating,mean_vehicle_age,median_days_to_report,crash_rate_pct
0,audi,a4,2016,Luxury,5.0,5.0,4.0800,14.0,0.00
1,audi,a4,2017,Luxury,5.0,4.0,3.4512,18.5,4.88
2,audi,a4,2018,Luxury,5.0,4.0,5.1429,73.5,2.38
3,audi,a4,2020,Luxury,5.0,4.0,4.9134,33.0,0.79
4,bmw,3 series,2016,Luxury,5.0,4.0,4.6882,5.0,12.90


In [3]:
tr, te = train_test_split(d, test_size=TEST_SIZE, random_state=RANDOM_STATE)
keys = ['make','model','model_year']
print(f'train {len(tr)}  test {len(te)}  shared vehicles: {len(pd.merge(tr[keys], te[keys], on=keys))}')

train 163  test 55  shared vehicles: 0


## 2. Choosing features and the ridge penalty by 5-fold CV on the training set

alpha = 0 is plain least squares.

In [4]:
sets = {'ratings only': LAB, 'ratings+equip+segment': LAB+EQUIP+SEG,
        'ownership record only': RECORD, 'everything': LAB+EQUIP+SEG+RECORD}
kf = KFold(5, shuffle=True, random_state=RANDOM_STATE)
y = tr.crash_rate_pct.values
rows = []
for s, cols in sets.items():
    for a in [0, 1, 10, 30, 100, 300, 1000]:
        m = make_pipeline(StandardScaler(), LinearRegression() if a == 0 else Ridge(alpha=a))
        p = cross_val_predict(m, tr[cols].values, y, cv=kf)
        rows.append({'features': s, 'alpha': a, 'cv_rmse': np.sqrt(mean_squared_error(y, p))})
cv = pd.DataFrame(rows)
cv.pivot(index='features', columns='alpha', values='cv_rmse').round(3)

alpha,0,1,10,30,100,300,1000
features,,,,,,,
everything,5.312,5.269,5.081,4.928,4.863,5.005,5.253
ownership record only,5.099,5.097,5.081,5.068,5.095,5.216,5.380
ratings only,5.462,5.454,5.405,5.370,5.373,5.424,5.481
ratings+equip+segment,5.605,5.562,5.396,5.263,5.173,5.234,5.376


## 3. Final linear models, scored once on the test set

In [5]:
cols = LAB+EQUIP+SEG+RECORD
ridge = make_pipeline(StandardScaler(), Ridge(alpha=100)).fit(tr[cols], y)
ols = LinearRegression().fit(tr[cols], y)
yt = te.crash_rate_pct.values
out = []
for name, p in [('ridge alpha=100 (chosen)', ridge.predict(te[cols])), ('LinearRegression', ols.predict(te[cols])),
                ('baseline: train mean', np.full(len(yt), y.mean()))]:
    out.append({'model': name, 'RMSE': np.sqrt(mean_squared_error(yt, p)), 'MAE': mean_absolute_error(yt, p), 'R2': r2_score(yt, p)})
pd.DataFrame(out).round(3)

,model,RMSE,MAE,R2
0,ridge alpha=100 (chosen),3.387,2.564,-0.025
1,LinearRegression,4.181,3.097,-0.563
2,baseline: train mean,3.578,3.065,-0.144


In [6]:
# coefficients of the chosen model in original units (pct points per unit)
per_unit = ridge[-1].coef_ / ridge[0].scale_
pd.Series(per_unit, index=cols).round(3).sort_values()

seg_Luxury                -2.035
seg_Minivan               -1.923
OverallFrontCrashRating   -1.655
log_complaints            -0.860
log_recalls               -0.501
mean_vehicle_age          -0.470
RolloverRating            -0.426
has_fcw                   -0.409
seg_Truck                 -0.160
rollover_prob_pct         -0.116
median_days_to_report     -0.066
has_ldw                    0.319
OverallRating              0.672
seg_Sedan                  1.421
seg_Electric               2.431
OverallSideCrashRating     2.658
seg_Hybrid                 3.955
dtype: float64

## 4. Logistic regression on the Naive Bayes data and split

In [7]:
c = pd.read_csv('../data/clean/complaints_clean.csv.gz', low_memory=False)
c['summary'] = c['summary'].fillna('').astype(str)
c = c[c['summary'].str.split().str.len() >= 5]
c['label'] = np.where(c['crash'], 'Crash', 'No crash')
df = c[['odiNumber','summary','label']].reset_index(drop=True)
train, test = train_test_split(df, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=df['label'])
assert not set(train.odiNumber) & set(test.odiNumber)
vec = CountVectorizer(stop_words='english', min_df=5, max_features=6000)
Xtr, Xte = vec.fit_transform(train.summary), vec.transform(test.summary)   # vocabulary from TRAIN only
ytr, yte = train.label.values, test.label.values
print(f'train {Xtr.shape}  test {Xte.shape}')

train (62461, 6000)  test (20821, 6000)


In [8]:
skf = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
rows = []
for cw in (None, 'balanced'):
    for C in [0.01, 0.1, 0.3, 1.0, 3.0]:
        p = cross_val_predict(LogisticRegression(C=C, class_weight=cw, solver='liblinear', max_iter=2000),
                              Xtr, ytr, cv=skf, n_jobs=-1)
        rows.append({'class_weight': cw or 'none', 'C': C, 'cv_acc': accuracy_score(ytr, p),
                     'cv_f1_crash': f1_score(ytr, p, pos_label='Crash')})
pd.DataFrame(rows).round(4)

,class_weight,C,cv_acc,cv_f1_crash
0,none,0.01,0.9715,0.5928
1,none,0.10,0.9760,0.6954
2,none,0.30,0.9766,0.7118
3,none,1.00,0.9758,0.7138
4,none,3.00,0.9732,0.6939
5,balanced,0.01,0.9358,0.5564
6,balanced,0.10,0.9471,0.5949
7,balanced,0.30,0.9509,0.6074
8,balanced,1.00,0.9531,0.6098
9,balanced,3.00,0.9543,0.6076


In [9]:
logit = LogisticRegression(C=1.0, solver='liblinear', max_iter=2000).fit(Xtr, ytr)
models = {'logistic (C=1)': logit, 'Multinomial NB': MultinomialNB(alpha=1.0).fit(Xtr, ytr),
          'Bernoulli NB': BernoulliNB(alpha=1.0, binarize=0.5).fit(Xtr, ytr)}
rows = []
for name, m in models.items():
    p = m.predict(Xte)
    rows.append({'model': name, 'accuracy': accuracy_score(yte, p),
                 'precision': precision_score(yte, p, pos_label='Crash'),
                 'recall': recall_score(yte, p, pos_label='Crash'), 'f1': f1_score(yte, p, pos_label='Crash')})
pd.DataFrame(rows).round(4)

,model,accuracy,precision,recall,f1
0,logistic (C=1),0.9784,0.8303,0.6839,0.7500
1,Multinomial NB,0.9441,0.4515,0.8298,0.5848
2,Bernoulli NB,0.9517,0.4941,0.8055,0.6125


In [10]:
pd.DataFrame(confusion_matrix(yte, logit.predict(Xte), labels=['No crash','Crash']),
             index=['actual No crash','actual Crash'], columns=['pred No crash','pred Crash'])

,pred No crash,pred Crash
actual No crash,19696,138
actual Crash,312,675


## 5. Reading a coefficient as an odds multiplier

In [11]:
terms = vec.get_feature_names_out()
sign = 1 if list(logit.classes_).index('Crash') == 1 else -1
coef = pd.Series(sign * logit.coef_[0], index=terms)
b0 = sign * logit.intercept_[0]
b = coef['pole']
p0 = 1 / (1 + np.exp(-b0)); p1 = 1 / (1 + np.exp(-(b0 + b)))
print(f"'pole': coefficient {b:.3f}, odds multiplier {np.exp(b):.2f}")
print(f'probability with no informative words {p0:.3f}; with one mention of pole {p1:.3f}')
np.exp(coef).sort_values().iloc[list(range(6)) + list(range(-6, 0))].round(2)

'pole': coefficient 2.269, odds multiplier 9.67
probability with no informative words 0.026; with one mention of pole 0.208


pothole          0.12
extinguished     0.15
paint            0.16
fortunately      0.19
reset            0.19
lead             0.20
surged           8.69
tree             8.99
pole             9.67
guardrail        9.82
collided        24.71
crashed         32.33
dtype: float64